## implementation of BFS solver

exhaustive search of possible grid states. find shortest path from starting state to target state (red car exits)

In [63]:
puzzles_6 = []
puzzles_6_op_moves = []

# load 6x6 puzzles with no walls
with open("../data/rush_no_walls.txt", "r", encoding="utf-8") as fin:
    for line in fin:
        stripped = line.strip()
        if not stripped:
            continue

        parts = stripped.split(maxsplit=2)
        if len(parts) != 3:
            continue

        moves, board_desc, _ = parts
        puzzles_6.append(board_desc)
        puzzles_6_op_moves.append(moves)

print(puzzles_6[0])

GBBoLoGHIoLMGHIAAMCCCKoMooJKDDEEJFFo


In [55]:
puzzles_4 = []
puzzles_4_op_moves = []

# load 4x4 puzzles with no walls
with open("../data/rush_4.txt", "r", encoding="utf-8") as fin:
    for line in fin:
        stripped = line.strip()
        if not stripped:
            continue

        parts = stripped.split(" ")
        moves, board = parts

        puzzles_4.append(board)
        puzzles_4_op_moves.append(moves)

print(puzzles_4[0])
print(puzzles_4_op_moves[0])

BAoooEDFoooooCoo
4


In [43]:
def generate_next_states(grid_str: str) -> list[str]:
    """
    returns all valid possible moves for input grid state
    """
    n = int(len(grid_str) ** 0.5)
    if n * n != len(grid_str):
        raise ValueError("grid_str length must be a square")

    grid = [list(grid_str[i * n:(i + 1) * n]) for i in range(n)]

    # occupied cells for each piece (A-Z, excluding 'o')
    pieces = {}
    for r in range(n):
        for c in range(n):
            ch = grid[r][c]
            if ch != "o":
                pieces.setdefault(ch, []).append((r, c))

    def orientation(cells):
        rows = {r for r, _ in cells}
        cols = {c for _, c in cells}
        if len(cells) == 1:
            return "single"
        if len(rows) == 1:
            return "h"   # horizontal
        if len(cols) == 1:
            return "v"   # vertical
        raise ValueError(f"invalid piece shape for cells: {cells}")

    def can_move(cells_set, dr, dc, step):
        for r, c in cells_set:
            nr, nc = r + dr * step, c + dc * step
            if not (0 <= nr < n and 0 <= nc < n):
                return False
            if grid[nr][nc] != "o" and (nr, nc) not in cells_set:
                return False
        return True

    def apply_move(piece, cells, dr, dc, step):
        new_grid = [row[:] for row in grid]
        for r, c in cells:
            new_grid[r][c] = "o"
        for r, c in cells:
            nr, nc = r + dr * step, c + dc * step
            new_grid[nr][nc] = piece
        return "".join("".join(row) for row in new_grid)

    results = []
    seen = set()

    for piece in sorted(pieces.keys()):
        cells = pieces[piece]
        cells_set = set(cells)
        orient = orientation(cells)

        directions = []
        if orient in ("h", "single"):
            directions.extend([(0, -1), (0, 1)])
        if orient in ("v", "single"):
            directions.extend([(-1, 0), (1, 0)])

        for dr, dc in directions:
            step = 1
            while can_move(cells_set, dr, dc, step):
                state = apply_move(piece, cells, dr, dc, step)
                if state not in seen:
                    seen.add(state)
                    results.append(state)
                step += 1

    return results

# test grid states
grid_state = "AAooooCoo"
states = generate_next_states(grid_state)
print(states)

['oAAoooCoo', 'AAoooooCo', 'AAooooooC', 'AAoCooooo']


In [65]:
from collections import deque

# 4x4 puzzle solver
def bfs_path_to_target(grid_str: str, exit_row: int) -> list[str] | None:
    """
    Find a shortest path from grid_str to any target state using BFS.
    A target state is any state where at least one 'A' is on the boundary.
    Returns the path as a list of states [start, ..., target], or None if unreachable.
    """
    n = int(len(grid_str) ** 0.5)
    if n * n != len(grid_str):
        raise ValueError("grid_str length must be a square")

    def is_target(state: str) -> bool:
        for i, ch in enumerate(state):
            if ch != "A":
                continue
            r, c = divmod(i, n)
            if r == exit_row and c == n-1: # end of exit row   ; or r == n - 1 or c == 0 or c == n - 1:
                return True
        return False

    q = deque([grid_str])
    parent = {grid_str: None}  # also acts as visited set

    while q:
        cur = q.popleft()

        if is_target(cur):
            path = []
            node = cur
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1]

        for nxt in generate_next_states(cur):
            if nxt not in parent:
                parent[nxt] = cur
                q.append(nxt)

    return None

exit_row = 2 # use exit row 1 for 4x4 grid and exit row 2 for 6x6
for puzzle, moves in zip(puzzles_6[:10], puzzles_6_op_moves[:10]):
    sol = bfs_path_to_target(puzzle, exit_row)

    # print(f"solution: {len(sol)-1}, expected: {moves}")
    #print(f"got path: {sol}")
    #print("----")
    if len(sol)-1 > int(moves):
        print("WRONG ANS")